# Physiology–MWL correlations by phase and group

Exploratory block-level Spearman correlations within each group × phase. Training phases contain repeated blocks per participant, so p-values are descriptive, not confirmatory independent-observation inference. Pre-Test is pre-training, Tests 1–3 are during training, and Evaluation is post-training without concurrent haptic exposure. Group differences in coefficients and changes across phases are not formally tested.

## Imports, configuration, and paths

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
pd.set_option("display.max_columns",100); pd.set_option("display.width",190)
CORRELATION_METHOD="spearman"; INCLUDE_IMPUTED_MWL=True; FDR_ALPHA=.05
SAVE_FIGURES=True; SAVE_TABLES=True; TOP_N=10
PHASE_ORDER=["pre_test","test_1","test_2","test_3","evaluation"]
GROUP_ORDER=["Haptic","NoHA"]; MODALITY_ORDER=["ECG","EDA","RESP","TEMP","fNIRS"]
assert CORRELATION_METHOD=="spearman"
def find_root(start=None):
    start=Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start,*start.parents):
        if (candidate/"outputs/final_features/physiology_mwl_analysis_dataset.csv").exists(): return candidate
    raise FileNotFoundError("Repository root not found")
ROOT=find_root(); INPUT=ROOT/"outputs/final_features/physiology_mwl_analysis_dataset.csv"
FIG=ROOT/"postprocessing/outputs/figures"; TAB=ROOT/"postprocessing/outputs/tables"
FIG.mkdir(parents=True,exist_ok=True); TAB.mkdir(parents=True,exist_ok=True)
print("Input:",INPUT)

## Load and validate

The feature identification reproduces notebook 01: only approved modality prefixes are used, never arbitrary numeric columns.

In [ ]:
all_data=pd.read_csv(INPUT)
PREFIX={"ECG":"delta_ecg_","EDA":"delta_eda_","RESP":"delta_resp_","TEMP":"delta_temp_","fNIRS":"fnirs_"}
modality_features={m:[c for c in all_data if c.startswith(p)] for m,p in PREFIX.items()}
features=[f for m in MODALITY_ORDER for f in modality_features[m]]
feature_modality={f:m for m,fs in modality_features.items() for f in fs}
assert len(features)==79 and len(set(features))==79
assert set(all_data.phase)==set(PHASE_ORDER) and set(all_data.group)==set(GROUP_ORDER)
assert not all_data.duplicated(["participant_id","phase","block_index"]).any()
data=all_data.copy() if INCLUDE_IMPUTED_MWL else all_data[all_data.mwl_source.ne("imputed_previous")].copy()
family_qc=data.groupby(["phase","group"]).agg(observations=("mwl_value","size"),participants=("participant_id","nunique"),n_unique_mwl=("mwl_value","nunique"),mwl_min=("mwl_value","min"),mwl_max=("mwl_value","max")).reindex(pd.MultiIndex.from_product([PHASE_ORDER,GROUP_ORDER],names=["phase","group"])).reset_index()
print("Rows",len(data),"participants",data.participant_id.nunique(),"features",len(features))
print(family_qc.to_string(index=False))

## Correlations and ten independent FDR families

Pairwise-complete observations are used feature by feature. Undefined tests are retained and excluded from that family's Benjamini–Hochberg denominator.

In [ ]:
def bh(values):
    p=np.asarray(values,float); out=np.full(p.shape,np.nan); mask=np.isfinite(p)
    if not mask.any(): return out
    v=p[mask]; order=np.argsort(v); ranked=v[order]; n=len(v)
    q=np.minimum.accumulate((ranked*n/np.arange(1,n+1))[::-1])[::-1]
    restored=np.empty(n); restored[order]=np.clip(q,0,1); out[np.flatnonzero(mask)]=restored
    return out
def one(frame,phase,group,feature):
    d=frame[["participant_id","mwl_value",feature]].dropna()
    base=dict(phase=phase,group=group,feature=feature,modality=feature_modality[feature],n_observations=len(d),n_participants=d.participant_id.nunique(),n_unique_mwl=d.mwl_value.nunique(),mwl_min=d.mwl_value.min() if len(d) else np.nan,mwl_max=d.mwl_value.max() if len(d) else np.nan)
    if len(d)<3 or d.participant_id.nunique()<2: return {**base,"rho":np.nan,"p_value":np.nan,"status":"insufficient_data"}
    if d.mwl_value.nunique()<2: return {**base,"rho":np.nan,"p_value":np.nan,"status":"constant_mwl"}
    if d[feature].nunique()<2: return {**base,"rho":np.nan,"p_value":np.nan,"status":"constant_feature"}
    r=spearmanr(d[feature],d.mwl_value,nan_policy="omit")
    return {**base,"rho":float(r.statistic),"p_value":float(r.pvalue),"status":"ok"}
def calculate(frame):
    families=[]
    for phase in PHASE_ORDER:
      for group in GROUP_ORDER:
        d=frame[(frame.phase==phase)&(frame.group==group)]
        r=pd.DataFrame([one(d,phase,group,f) for f in features]); r["p_fdr"]=bh(r.p_value)
        r["significant_nominal"]=r.p_value.lt(FDR_ALPHA); r["significant_fdr"]=r.p_fdr.lt(FDR_ALPHA); families.append(r)
    return pd.concat(families,ignore_index=True)
results=calculate(data)
results=results[["phase","group","feature","modality","n_observations","n_participants","n_unique_mwl","mwl_min","mwl_max","rho","p_value","p_fdr","significant_nominal","significant_fdr","status"]]
assert len(results)==790
if SAVE_TABLES: results.to_csv(TAB/"physiology_mwl_spearman_by_phase_group.csv",index=False)
print(results.groupby(["phase","group","status"]).size().to_string())

## Family and modality summary

In [ ]:
summary_rows=[]
for phase in PHASE_ORDER:
 for group in GROUP_ORDER:
  family=results[(results.phase==phase)&(results.group==group)]; source=data[(data.phase==phase)&(data.group==group)]
  for modality in ["ALL",*MODALITY_ORDER]:
   s=family if modality=="ALL" else family[family.modality==modality]; v=s[s.status=="ok"]
   pos_idx=v.rho.idxmax() if len(v) else None; neg_idx=v.rho.idxmin() if len(v) else None
   pos=(v.loc[pos_idx,"feature"],v.loc[pos_idx,"rho"]) if pos_idx is not None else ("",np.nan); neg=(v.loc[neg_idx,"feature"],v.loc[neg_idx,"rho"]) if neg_idx is not None else ("",np.nan); idx=v.rho.abs().idxmax() if len(v) else None
   summary_rows.append(dict(phase=phase,group=group,modality=modality,observations=len(source),participants=source.participant_id.nunique(),valid_correlations=len(v),nominal_p_lt_0_05=int(s.significant_nominal.sum()),fdr_q_lt_0_05=int(s.significant_fdr.sum()),strongest_positive_feature=pos[0],strongest_positive_rho=pos[1],strongest_negative_feature=neg[0],strongest_negative_rho=neg[1],strongest_absolute_feature=v.loc[idx,"feature"] if idx is not None else "",strongest_absolute_rho=v.loc[idx,"rho"] if idx is not None else np.nan,fdr_modalities=";".join(sorted(s.loc[s.significant_fdr,"modality"].unique()))))
summary=pd.DataFrame(summary_rows)
if SAVE_TABLES: summary.to_csv(TAB/"physiology_mwl_correlation_phase_summary.csv",index=False)
print(summary.to_string(index=False))

## Descriptive Haptic–NoHA rho differences

These coefficient differences have no p-value and are not tests of a group difference.

In [ ]:
h=results[results.group=="Haptic"][["phase","feature","modality","rho","n_observations","n_participants","status"]].rename(columns={"rho":"rho_haptic","n_observations":"n_haptic_observations","n_participants":"n_haptic_participants","status":"status_haptic"})
n=results[results.group=="NoHA"][["phase","feature","rho","n_observations","n_participants","status"]].rename(columns={"rho":"rho_noha","n_observations":"n_noha_observations","n_participants":"n_noha_participants","status":"status_noha"})
diff=h.merge(n,on=["phase","feature"],validate="one_to_one"); diff["delta_rho"]=diff.rho_haptic-diff.rho_noha; diff["abs_delta_rho"]=diff.delta_rho.abs(); diff["comparison_status"]=np.where(diff[["rho_haptic","rho_noha"]].notna().all(axis=1),"descriptive_available","undefined_source_correlation")
if SAVE_TABLES: diff.to_csv(TAB/"physiology_mwl_haptic_noha_rho_differences_by_phase.csv",index=False)
for phase in PHASE_ORDER:
 print("\n",phase,"largest descriptive differences\n",diff[diff.phase==phase].nlargest(TOP_N,"abs_delta_rho")[["feature","modality","rho_haptic","rho_noha","delta_rho","abs_delta_rho"]].to_string(index=False))

## Matched Haptic and NoHA heatmaps

In [ ]:
boundaries=np.cumsum([len(modality_features[m]) for m in MODALITY_ORDER])[:-1]; starts=np.r_[0,boundaries]; ends=np.r_[boundaries,len(features)]
fig,axes=plt.subplots(1,2,figsize=(14,22),sharey=True)
for ax,group in zip(axes,GROUP_ORDER):
 s=results[results.group==group]; matrix=s.pivot(index="feature",columns="phase",values="rho").reindex(index=features,columns=PHASE_ORDER); sig=s.pivot(index="feature",columns="phase",values="significant_fdr").reindex(index=features,columns=PHASE_ORDER)
 image=ax.imshow(matrix.to_numpy(float),aspect="auto",cmap="RdBu_r",vmin=-1,vmax=1); ax.set_xticks(range(5),PHASE_ORDER,rotation=25,ha="right"); ax.set_title(group)
 for i in range(79):
  for j in range(5):
   if bool(sig.iloc[i,j]): ax.text(j,i,"★",ha="center",va="center",fontsize=8)
 for b in boundaries: ax.axhline(b-.5,color="black",linewidth=1.2)
axes[0].set_yticks(range(79),features,fontsize=5)
for m,a,b in zip(MODALITY_ORDER,starts,ends): axes[0].text(-.53,(a+b-1)/2,m,transform=axes[0].get_yaxis_transform(),ha="right",va="center",fontsize=8,fontweight="bold")
fig.suptitle("Physiology–MWL Spearman rho by phase and group\n★ family-specific FDR q < 0.05",y=.995); fig.colorbar(image,ax=axes,fraction=.018,pad=.025,label="Spearman rho"); fig.subplots_adjust(left=.32,right=.92,top=.96,bottom=.05,wspace=.08)
if SAVE_FIGURES: fig.savefig(FIG/"physiology_mwl_correlations_by_phase_group.png",dpi=200,bbox_inches="tight")
plt.show()

## Descriptive rho-difference heatmap

In [ ]:
matrix=diff.pivot(index="feature",columns="phase",values="delta_rho").reindex(index=features,columns=PHASE_ORDER)
fig,ax=plt.subplots(figsize=(9,22)); image=ax.imshow(matrix.to_numpy(float),aspect="auto",cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(5),PHASE_ORDER,rotation=25,ha="right"); ax.set_yticks(range(79),features,fontsize=5); ax.set_title("Descriptive Haptic–NoHA difference in Spearman rho\n(no formal coefficient-difference test)")
for b in boundaries: ax.axhline(b-.5,color="black",linewidth=1.2)
for m,a,b in zip(MODALITY_ORDER,starts,ends): ax.text(-.53,(a+b-1)/2,m,transform=ax.get_yaxis_transform(),ha="right",va="center",fontsize=8,fontweight="bold")
fig.colorbar(image,ax=ax,fraction=.025,pad=.03,label="rho_Haptic − rho_NoHA"); fig.subplots_adjust(left=.43,right=.91,top=.96,bottom=.05)
if SAVE_FIGURES: fig.savefig(FIG/"physiology_mwl_haptic_noha_rho_difference_by_phase.png",dpi=200,bbox_inches="tight")
plt.show()

## Imputed-MWL sensitivity

In [ ]:
with_imp=calculate(all_data)[["phase","group","feature","rho"]].rename(columns={"rho":"rho_with_imputation"}); observed=calculate(all_data[all_data.mwl_source!="imputed_previous"])[["phase","group","feature","rho"]].rename(columns={"rho":"rho_observed_only"})
sensitivity=with_imp.merge(observed,on=["phase","group","feature"],validate="one_to_one"); sensitivity["absolute_change_rho"]=(sensitivity.rho_with_imputation-sensitivity.rho_observed_only).abs(); finite=sensitivity.absolute_change_rho.dropna(); max_change=finite.max(); median_change=finite.median(); largest=sensitivity.loc[sensitivity.absolute_change_rho.idxmax()]
if SAVE_TABLES: sensitivity.to_csv(TAB/"physiology_mwl_phase_group_imputation_sensitivity.csv",index=False)
print("Maximum |rho change|",max_change,"median",median_change); print(largest.to_string())

## Temporal-pattern and automated family report

Similarity, opposite-direction, and large-difference listings are descriptive aids only.

In [ ]:
valid=diff.dropna(subset=["rho_haptic","rho_noha"]); similar=valid.groupby("feature").agg(modality=("modality","first"),mean_abs_delta=("abs_delta_rho","mean"),max_abs_delta=("abs_delta_rho","max")).sort_values("mean_abs_delta").head(10); opposite=valid[(valid.rho_haptic*valid.rho_noha<0)&(valid.rho_haptic.abs()>=.2)&(valid.rho_noha.abs()>=.2)].sort_values("abs_delta_rho",ascending=False)
print("Most similar group profiles across phases:\n",similar.to_string()); print("\nOpposite directions with both |rho| >= .20:\n",opposite[["phase","feature","modality","rho_haptic","rho_noha","delta_rho"]].to_string(index=False) if len(opposite) else "None")
overall=summary[summary.modality=="ALL"]
for phase in PHASE_ORDER:
 print("\n"+phase)
 for group in GROUP_ORDER:
  row=overall[(overall.phase==phase)&(overall.group==group)].iloc[0]; fam=results[(results.phase==phase)&(results.group==group)&(results.status=="ok")]; top=fam.assign(a=fam.rho.abs()).nlargest(5,"a")
  print(f" {group}: observations={row.observations}, participants={row.participants}, valid={row.valid_correlations}, nominal={row.nominal_p_lt_0_05}, FDR={row.fdr_q_lt_0_05}; strongest: "+"; ".join(f"{r.feature} ({r.rho:+.3f})" for _,r in top.iterrows()))
for phase in ["pre_test","evaluation"]:
 for group in GROUP_ORDER:
  q=family_qc[(family_qc.phase==phase)&(family_qc.group==group)].iloc[0]; print(f"SMALL-N CAUTION: {phase} {group}: N={q.observations}, participants={q.participants}, MWL={q.mwl_min:g}–{q.mwl_max:g}; exploratory and potentially unstable.")
print(f"Imputation sensitivity maximum |rho change|={max_change:.6g}, median={median_change:.6g}.")
print("No Haptic–NoHA coefficient difference and no temporal coefficient change was formally tested.")